# 10 - Train a Supervised-Policy Model Offline

This notebook is the ranking check: can the model copy expert `argmax(Q*)`?

It uses the same data, embedder, and backbone as `05_train_offline_sv.ipynb`, but swaps the value head and `SvObjective` for `DiscreteActionHead` and `SpObjective`. The head is trained with hard cross-entropy onto a random argmax of `info_q_star` (ties broken uniformly) while the episode is running (`episode_done == 0`). There is no Bellman backup, no target network, and no magnitude regression.

1. Load previously collected `Datastore` streams from the Hub (must include `info_q_star`).
2. Build a `DataLoader` that samples fixed-length sequences from those streams.
3. Assemble a `Model` from an embedder, a backbone, and an action head.
4. Train with `SpObjective` and save with `push_model_to_hub`.

`Augmenter` remaps `action` ids and, via `input_vector_field` /
`output_vector_field` on `info_q_star`, reorders the Q vector with the **same**
permutation so it stays aligned with the remapped ids.

This is a short usage example, not a full experiment. Evaluate a saved checkpoint in `09_inference.ipynb`.


In [ ]:
import torch

from mouse_core import AdamW, AdamWFp32
from mouse_core.data import (
    DataLoader,
    Augmenter,
    Selector,
    NumericTokenizer,
    compose,
    load_stores_from_hub,
)
from mouse_core.objectives import SpObjective
from mouse_core.models import Model, preferred_dtype, push_model_to_hub
from mouse_core.models.backbone import Qwen3Backbone
from mouse_core.models.embedding import NumericEmbedder
from mouse_core.models.heads import DiscreteActionHead


DATASET_ID = "mouse-example-dataset"          # Hugging Face dataset repo for load_stores_from_hub
MODEL_ID = "mouse-example-model-offline-sp"      # Hugging Face model repo for push_model_to_hub
MAX_ACTIONS = 4                               # number of discrete actions predicted by the head
MAX_OBS_DISCRETE = 64                         # vocabulary size for discrete observations
SEQUENCE_LENGTH = 512                         # replay sequence length sampled by DataLoader
BATCH_SIZE = 4                                # sequences per optimizer step
NUM_CYCLES = 2                               # outer train cycles (print cadence)
TRAIN_STEPS = 50                             # optimizer updates per cycle (passed to run_train)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## Load Data

`load_stores_from_hub` downloads the dataset snapshot and reconstructs the saved `Datastore` objects. Each returned store is one ordered environment stream.


In [ ]:
stores = load_stores_from_hub(repo_id=DATASET_ID, split='train', force_download=True)

## Data pipeline

`DataLoader` samples contiguous windows up to `sequence_length` (a max) from one or more datastores. Each sequence may be shorter than the max depending on where the window starts in the store.

Pipeline order: `augmenter → selector → tokenizer → pack → embedder`.

| Stage | Role |
| --- | --- |
| **Augmenter** | `dict → dict` (`fields=` value transforms; `seed_field=` for shared draws within a `reseed` generation). Action permute sets `input_vector_field` / `output_vector_field` on `info_q_star` so Q* stays aligned. |
| **Selector** | `dict → dict` (`fields=` `input_field`/`output_field` keep/rename; keep `info_q_star`) |
| **Tokenizer** | `dict → StepTokens` (`input_field` / `output_field`; `objective_fields=` is `episode_done` / `info_q_star`; `grouping_field=`) |

Compose `train_transform = compose(augmenter, selector, tokenizer)`.
`DataLoader(transform=train_transform)` maps each step and packs into a `TokenBatch`.
Live inference in `09_inference.ipynb` uses the tokenizer without the augmenter so chosen actions match the env.


In [ ]:
# Pipeline order: augmenter → selector → tokenizer
# input_vector_field/output_vector_field share the action permute so Q*[perm[a]] stays aligned.

augmenter = Augmenter(
    seed_field="task_index",
    fields=[
        {
            "type": "discrete",
            "input_field": "action",
            "input_vector_field": "info_q_star",
            "vocab_size": MAX_ACTIONS,
            "permute": True,
        },
        {
            "type": "discrete",
            "input_field": "observation",
            "vocab_size": MAX_OBS_DISCRETE,
            "permute": True,
        },
    ],
)

selector = Selector(
    fields=[
        {
            "input_field": "action",
        },
        {
            "input_field": "observation",
        },
        {
            "input_field": "reward",
        },
        {
            "input_field": "episode_done",
        },
        {
            "input_field": "task_done",
        },
        {
            "input_field": "info_q_star",
        },
        {
            "input_field": "task_index",
        },
    ],
)

tokenizer = NumericTokenizer(
    input_fields=[
        {
            "type": "discrete",
            "input_field": "action",
        },
        {
            "type": "discrete",
            "input_field": "observation",
        },
        {
            "type": "fourier",
            "input_field": "reward",
        },
        {
            "type": "discrete",
            "input_field": "episode_done",
        },
    ],
    objective_fields=[
        {
            "input_field": "episode_done",
        },
        {
            "input_field": "info_q_star",
        },
    ],
    grouping_field="task_index",
)

train_transform = compose(augmenter, selector, tokenizer)

loader = DataLoader(
    stores=stores,
    sequence_length=SEQUENCE_LENGTH,
    batch_size=BATCH_SIZE,
    transform=train_transform,
    prefetch=4,
    num_workers=0,
)


## Build The Model

A Mouse Core `Model` has three main pieces:

- `NumericEmbedder` maps a tokenized `TokenBatch` (modalities keyed by name; add `vocab_size` / `std` here) into vectors.
- `Qwen3Backbone` processes those tokens with a transformer backbone.
- `DiscreteActionHead` predicts one logit per discrete action.

The backbone exposes `hidden_dim`, and the embedder and head use that same value so the pieces connect cleanly.

`NumericEmbedder` modality types used here:

- `discrete` for integer IDs such as actions, observations, and episode/task done codes.
- `fourier` for scalar numeric values such as rewards.
- `learnable` when you want extra learned tokens that are not tied to a row field.

`Model(...)` wraps the pieces behind a single forward call that returns predictions, objective data, and an optional cache.


In [ ]:
backbone = Qwen3Backbone(pretrained="Qwen/Qwen3-0.6B")

encoder = NumericEmbedder(
    hidden_dim=backbone.hidden_dim,
    modalities=[
        {
            "type": "discrete",
            "field": "action",
            "vocab_size": MAX_ACTIONS,
            "std": 0.02,
        },
        {
            "type": "discrete",
            "field": "observation",
            "vocab_size": MAX_OBS_DISCRETE,
            "std": 0.02,
        },
        {
            "type": "fourier",
            "field": "reward",
            "std": 0.02,
        },
        {
            "type": "discrete",
            "field": "episode_done",
            "vocab_size": 3,
            "std": 0.02,
        },
    ],
)

head = DiscreteActionHead(
    in_features=backbone.hidden_dim,
    out_features=MAX_ACTIONS,
    hidden_dim=backbone.hidden_dim,
    num_layers=1,
    scale=0.1,
)

model = Model(encoder=encoder, backbone=backbone, heads=head).train().to(device=device, dtype=preferred_dtype(device))
print(model)


## Training Phase

Each outer cycle runs `TRAIN_STEPS` optimizer updates via `run_train`.

1. `inputs, objective_data = loader.next_batch()` samples ragged step windows (up to `SEQUENCE_LENGTH`).
2. `model(inputs)` embeds the `TokenBatch`, runs the backbone, and produces flat per-step head predictions.
3. `SpObjective` trains `predictions["action"]` with hard CE onto a uniformly random argmax of `objective_data["info_q_star"]` (ties are not biased toward the lowest action index).
4. `AdamW` updates weights. Swap in `AdamWFp32` for fp32 masters on bf16 weights. There is no Polyak / target network.

`SpObjective(mask_key="episode_done")` drops any step where `episode_done != 0` (terminated or truncated): the episode is over, so there is no next action to imitate. Attention is task-isolated by `task_index` so the backbone cannot mix maps.


In [ ]:
optimizer = AdamW(model.parameters(), lr=1e-05, weight_decay=0.0, betas=(0.9, 0.95), eps=1e-08)
objective = SpObjective(loss_type="ce", predictions_key="action", targets_key="info_q_star", mask_key="episode_done")

def run_train(*, model: Model, optimizer: AdamW | AdamWFp32, objective: SpObjective, loader: DataLoader, num_steps: int) -> tuple[torch.Tensor, dict[str, float]]:
    """Run ``num_steps`` optimizer steps on batches from ``loader``."""
    model.train()
    loss: torch.Tensor | None = None
    metrics: dict[str, float] = {}
    for _ in range(num_steps):
        inputs, objective_data = loader.next_batch()
        predictions, _ = model(inputs)
        loss, metrics = objective(objective_data.to(device), predictions)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    assert loss is not None
    return (loss, metrics)


## Run

Each of `NUM_CYCLES` cycles calls `run_train(num_steps=TRAIN_STEPS)`. Score the checkpoint later in `09_inference.ipynb`.


In [ ]:
for cycle in range(NUM_CYCLES):
    loss, metrics = run_train(model=model, optimizer=optimizer, objective=objective, loader=loader, num_steps=TRAIN_STEPS)
    print(f"cycle={cycle} train  loss={loss.item():.4f}  sp={metrics['action']:.4f}")
loader.close()


## Push To The Hub

`push_model_to_hub` saves the model architecture and weights together. Later, `load_model` can reconstruct the full `Model` without repeating the embedder, backbone, and head definitions.


In [ ]:
model.eval().to("cpu")
url = push_model_to_hub(model=model, repo_id=MODEL_ID, private=False, clear=True)
print(f"Pushed to {url}")